Model Check

In [6]:
import base64
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from io import BytesIO
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Load the OCR model and processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-printed')

def preprocess_image(image):
    img_gray = image.convert('L')

    width, height = img_gray.size
    crop_box = (int(width * 0.2), int(height * 0.10), int(width * 0.8), int(height * 0.99))
    img_zoomed = img_gray.crop(crop_box)

    enhancer = ImageEnhance.Contrast(img_zoomed)
    img_contrast = enhancer.enhance(1.5)

    img_smoothed = img_contrast.filter(ImageFilter.SMOOTH_MORE)
    img_sharp = img_smoothed.filter(ImageFilter.SHARPEN)
    img_rgb = img_sharp.convert('RGB')

    return img_rgb

def get_captcha_text_from_image(image_path):
    # Open the CAPTCHA image
    image = Image.open(image_path)

    # Preprocess the image
    preprocessed_image = preprocess_image(image)

    # Convert the preprocessed image to the format suitable for the OCR model
    pixel_values = processor(images=preprocessed_image, return_tensors="pt").pixel_values

    # Generate CAPTCHA text using the Hugging Face TrOCR model
    generated_ids = model.generate(pixel_values)
    
    # Extract recognized CAPTCHA text (case-sensitive)
    captcha_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    return captcha_text

# Example usage with the uploaded image
image_path = './preprocessed_captcha_image.png'  # Update this path with your image path
captcha_text = get_captcha_text_from_image(image_path)
print(f"Recognized CAPTCHA text: {captcha_text}")


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Recognized CAPTCHA text: X6G3PA


In [ ]:
<h2>
	<a onclick="return startCommitRequest();" href="extern/appointment_showMonth.do?locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=03.09.2024" style="margin-left: 2em; margin-right: 2em;"><img src="images/go-previous.gif"></a>
	10/2024
	<a onclick="return startCommitRequest();" href="extern/appointment_showMonth.do?locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=03.11.2024" style="margin-left: 2em; margin-right: 2em;"><img src="images/go-next.gif"></a>
</h2>

Correct code

In [1]:
import time
import base64
from PIL import Image, ImageEnhance, ImageFilter
from io import BytesIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Load the OCR model and processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-printed')

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--enable-logging")
options.add_argument("--log-level=0")
options.add_argument("--no-sandbox")
options.add_argument("--lang=en")
options.add_argument("--disable-translate")
options.add_argument("--start-maximized")  # Open browser in full screen

# Create a Chrome service object and set up the driver
chrome_service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=chrome_service, options=options)

def preprocess_image(image):
    img_gray = image.convert('L')

    width, height = img_gray.size
    crop_box = (int(width * 0.2), int(height * 0.10), int(width * 0.8), int(height * 0.99))
    img_zoomed = img_gray.crop(crop_box)

    enhancer = ImageEnhance.Contrast(img_zoomed)
    img_contrast = enhancer.enhance(1.5)

    img_smoothed = img_contrast.filter(ImageFilter.SMOOTH_MORE)
    img_sharp = img_smoothed.filter(ImageFilter.SHARPEN)
    img_rgb = img_sharp.convert('RGB')

    return img_rgb

def get_captcha_text_from_image(driver, attempt):
    captcha_element = driver.find_element(By.XPATH, "//div[contains(@style, 'background:white url')]")
    
    captcha_base64 = captcha_element.get_attribute("style").split("base64,")[1].split("')")[0]
    
    if len(captcha_base64) % 4 != 0:
        captcha_base64 += '=' * (4 - len(captcha_base64) % 4)
    
    captcha_image = base64.b64decode(captcha_base64)
    
    image = Image.open(BytesIO(captcha_image)).convert("RGB")
    local_image_path = f"./downloaded_captcha_image_{attempt}.png"
    image.save(local_image_path)
    print(f"Original CAPTCHA image saved locally at {local_image_path}")

    preprocessed_image = preprocess_image(image)

    preprocessed_image_path = f"./preprocessed_captcha_image_{attempt}.png"
    preprocessed_image.save(preprocessed_image_path)
    print(f"Preprocessed CAPTCHA image saved locally at {preprocessed_image_path}")

    pixel_values = processor(images=preprocessed_image, return_tensors="pt").pixel_values

    generated_ids = model.generate(pixel_values)
    
    raw_output = processor.batch_decode(generated_ids)
    print(f"Raw OCR output: {raw_output}")
    
    captcha_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip().lower()

    print(f"Recognized CAPTCHA text: {captcha_text}")
    return captcha_text

def fill_and_submit_captcha(driver, captcha_text):
    captcha_input = driver.find_element(By.ID, "appointment_captcha_month_captchaText")
    captcha_input.clear()
    captcha_input.send_keys(captcha_text)
    
    continue_button = driver.find_element(By.ID, "appointment_captcha_month_appointment_showMonth")
    continue_button.click()

def check_if_on_next_page(driver):
    """ Checks if the page has navigated to the next step. """
    try:
        # Wait for the new page to load by looking for elements unique to the next page
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "global")))  # This element is present on the next page
        return True
    except:
        return False

def load_new_captcha(driver):
    refresh_button = driver.find_element(By.ID, "appointment_captcha_month_refreshcaptcha")
    refresh_button.click()

def main():
    driver.get("https://service2.diplo.de/rktermin/extern/appointment_showMonth.do?locationCode=kara&realmId=967&categoryId=1988")
    
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "appointment_captcha_month_captchaText")))

    attempts = 0
    max_attempts = 10

    while attempts < max_attempts:
        try:
            attempts += 1

            captcha_text = get_captcha_text_from_image(driver, attempts)

            fill_and_submit_captcha(driver, captcha_text)

            time.sleep(3)

            if check_if_on_next_page(driver):
                print("CAPTCHA was solved successfully! Moving to the next page.")
                break  # Break the loop when CAPTCHA is solved successfully
            else:
                print("CAPTCHA failed, retrying...")
                load_new_captcha(driver)  # Load a new CAPTCHA if it failed

        except Exception as e:
            print(f"An error occurred: {e}")
            load_new_captcha(driver)

    if attempts == max_attempts:
        print("Failed to solve CAPTCHA after several attempts.")
    
if __name__ == "__main__":
    main()


c:\Users\HNH TECH SOLUTIONS\Envs\captcha\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HNH TECH SOLUTIONS\Envs\captcha\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
VisionEncoderDecoderModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other r

Original CAPTCHA image saved locally at ./downloaded_captcha_image_1.png
Preprocessed CAPTCHA image saved locally at ./preprocessed_captcha_image_1.png


c:\Users\HNH TECH SOLUTIONS\Envs\captcha\Lib\site-packages\transformers\generation\utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Raw OCR output: ['</s>YMXGNX</s>']
Recognized CAPTCHA text: ymxgnx
CAPTCHA was solved successfully! Moving to the next page.


In [ ]:
https://service2.diplo.de/rktermin/extern/appointment_refreshCaptchamonth.do

In [ ]:
<div id="global">

	<div id="header">
	
		<div id="logo">
			<img src="style_images/auswaertiges-amt-logo-220x120.gif" alt="Auswärtiges Amt">
		</div>
		<div id="logo-app" style="background-image:  url('images/auswaertigesamt.gif'); min-height: 54px; width: 78%;">
			&nbsp;
		</div>
		<div id="nav-main" style="min-height: 28px;">
		

<ul>
    <li><a href="extern/appointment_showMonth.do?request_locale=de&amp;locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=" title="Diese Seite in Deutsch"><img src="images/flags/de.png" alt="Deutsche Fahne" title="Deutsche Fahne"></a></li>
    
        <li><a href="extern/appointment_showMonth.do?request_locale=en&amp;locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=" title="This page in English"><img src="images/flags/gb.png" alt="British Flag" title="British Flag"></a></li>
    
    
    
    
    
</ul>
		</div>		
	
	</div> <!-- end: #header -->			

	
	<div id="main" class="l-s">
			
		<div id="content">
		
			<div class="wrapper">	
			

<div style="font-size: 14pt; font-weight: bold; margin-bottom: 1em;">
	Booking of appointments for visa for language classes, pathway and bachelor programs
</div>



	
	
		<h2>
			Unfortunately, there are no appointments available at this time. New appointments will be made available for booking at regular intervals.
		</h2>
	
	





<h2>
	<a onclick="return startCommitRequest();" href="extern/appointment_showMonth.do?locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=03.09.2024" style="margin-left: 2em; margin-right: 2em;"><img src="images/go-previous.gif"></a>
	10/2024
	<a onclick="return startCommitRequest();" href="extern/appointment_showMonth.do?locationCode=kara&amp;realmId=967&amp;categoryId=1988&amp;dateStr=03.11.2024" style="margin-left: 2em; margin-right: 2em;"><img src="images/go-next.gif"></a>
</h2>



	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	
		<div style="width: 100%;">
			
		</div>
	






<script language="javascript" type="text/javascript">

$(function() {
	$("#commit-request").dialog({
		resizable : false,
		height : 200,
		width: 350,
		modal : true,
		autoOpen: false, 
		buttons : {
			/* none */
		},
		close: function( event, ui ) {
			$('#commit-request').dialog('open');
		}
	});
 	$( "#commit-request-progressbar" ).progressbar({
	  value: false
	});
});

function startCommitRequest(event) {
	
	$("#commit-request").css('cursor', 'progress');
	
	setTimeout("innerStartCommitRequest()", 500);

	return true;
	
}

function innerStartCommitRequest() {
	$('#commit-request').dialog('open');
}

</script>




			</div>
			
			<div class="bottom"></div>
			
		</div>
		
		<div id="context">
		
			<div class="wrapper">
				&nbsp;
			</div>
			
		</div>
		
	</div> <!-- end: #main -->
	<div id="footer">
		



<div style="min-height: 15px;">
	<ul>
		<li>RK-Termin&nbsp;1.3.0.4</li>
		<li style="margin-left: 5em;">
			
				<a href="extern/dsgvo.do?request_locale=en" target="_blank" title="Information on data protection and instructions for use"><img src="images/flags/gb.png" alt="Information on data protection and instructions for use" title="Information on data protection and instructions for use">&nbsp; Information
					on data protection and instructions for use</a>
			     </li>
	</ul>
</div>



	</div>
</div>